# 🚀 YOLOv8 + WandB Sweep (MLOps Demo)

이 노트북은 **Google Colab** 환경에서 YOLOv8과 Weights & Biases(WandB) Sweep을 활용한  
하이퍼파라미터 자동 최적화 파이프라인을 실습합니다.

---
### 📋 실행 순서
1. 환경 설치 (패키지)
2. GitHub 레포 클론 (또는 파일 업로드)
3. WandB 로그인 & API 키 설정
4. Baseline 학습
5. Sweep 설정 및 실행


## ① 환경 설정

In [1]:
# GPU 사용 여부 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU가 없습니다. 런타임 > 런타임 유형 변경 > GPU 선택 후 다시 실행하세요.")

CUDA available: True
GPU: Tesla T4


In [2]:
# 필수 패키지 설치
%pip install -q ultralytics wandb python-dotenv
print("✅ 패키지 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.9 MB/s eta 0:00:00
✅ 패키지 설치 완료


## ② 프로젝트 파일 불러오기

> **방법 A** — GitHub 레포가 있는 경우 아래 셀을 실행하세요.  
> **방법 B** — 파일을 직접 업로드하는 경우 방법 A 셀을 건너뛰고 방법 B를 실행하세요.

In [ ]:
# ── 방법 A: GitHub 레포 클론 ──────────────────────────────────────────
# 본인의 GitHub 레포 URL로 교체하세요.
REPO_URL = "https://github.com/jspark9703/mlops_demo.git"  # ← 수정

import os
if not os.path.exists("/content/mlops_demo"):
    !git clone {REPO_URL} /content/mlops_demo
    print("✅ 클론 완료")
else:
    !git -C /content/mlops_demo pull
    print("✅ 최신 코드로 업데이트")

os.chdir("/content/mlops_demo")
print(f"📂 작업 디렉토리: {os.getcwd()}")

In [ ]:
# ── 방법 B: 파일 직접 작성 (GitHub 레포 없을 경우) ──────────────────
# 방법 A를 사용한 경우 이 셀은 건너뛰세요.
import os
os.makedirs("/content/mlops_demo", exist_ok=True)
os.chdir("/content/mlops_demo")
print(f"📂 작업 디렉토리: {os.getcwd()}")

In [ ]:
%%writefile /content/mlops_demo/sweep.yaml
program: train_sweep.py
method: bayes
metric:
  goal: maximize
  name: metrics/mAP50-95(B)
parameters:
  batch:
    values: [8, 16, 32]
  epochs:
    values: [10, 15, 20]
  imgsz:
    values: [320, 480, 640]
  lr0:
    distribution: uniform
    min: 0.0001
    max: 0.01
  weight_decay:
    distribution: uniform
    min: 0.0001
    max: 0.001

In [ ]:
%%writefile /content/mlops_demo/train_baseline.py
import os
from pathlib import Path
from ultralytics import YOLO, settings
import wandb

# ClearML 자동 연동 해제
settings.update({'clearml': False})

def log_custom_plots(trainer):
    """
    에포크마다 검증 추론 이미지와 Loss 곡선을 WandB에 로깅합니다.
    """
    save_dir = Path(trainer.save_dir)

    pred_img_path = save_dir / "val_batch0_pred.jpg"
    if pred_img_path.exists():
        wandb.log({
            "Validation_Predictions": wandb.Image(
                str(pred_img_path),
                caption=f"Epoch {trainer.epoch}"
            )
        }, commit=False)

    results_path = save_dir / "results.png"
    if results_path.exists():
        wandb.log({
            "Loss & Metrics Plot": wandb.Image(
                str(results_path),
                caption=f"Results Plot up to Epoch {trainer.epoch}"
            )
        }, commit=False)

def main():
    wandb.init(project="mlops_demo_yolo", name="baseline_run", job_type="training")

    model = YOLO("yolov8n.pt")
    model.add_callback("on_fit_epoch_end", log_custom_plots)

    results = model.train(
        data="coco128.yaml",
        epochs=10,
        imgsz=640,
        batch=16,
        plots=True,
        save=True
    )

    wandb.finish()
    return results

if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mlops_demo/train_sweep.py
import os
from pathlib import Path
from ultralytics import YOLO, settings
import wandb

# ClearML 자동 연동 해제
settings.update({'clearml': False})

def log_epoch_metrics(trainer):
    """
    에포크마다 Loss, metrics, 검증 이미지를 WandB에 로깅합니다.
    """
    log_dict = {"epoch": trainer.epoch}

    if hasattr(trainer, "metrics") and trainer.metrics:
        log_dict.update(trainer.metrics)

    save_dir = Path(trainer.save_dir)
    pred_img_path = save_dir / "val_batch0_pred.jpg"
    if pred_img_path.exists():
        log_dict["Validation_Predictions"] = wandb.Image(
            str(pred_img_path),
            caption=f"Epoch {trainer.epoch}"
        )

    wandb.log(log_dict)

def main():
    default_config = {
        'lr0': 0.01,
        'weight_decay': 0.0005,
        'batch': 16,
        'epochs': 10,
        'imgsz': 640
    }

    with wandb.init(project="mlops_demo_yolo", job_type="sweep", config=default_config):
        config = wandb.config

        model = YOLO("yolov8n.pt")
        model.add_callback("on_fit_epoch_end", log_epoch_metrics)

        model.train(
            data="coco128.yaml",
            epochs=config.epochs,
            imgsz=config.imgsz,
            plots=True,
            save=True,
            lr0=config.lr0,
            weight_decay=config.weight_decay,
            batch=config.batch,
        )

if __name__ == "__main__":
    main()

## ③ WandB 로그인

In [ ]:
import wandb

# WandB API 키 입력 (최초 1회, 이후 캐시됨)
# https://wandb.ai/authorize 에서 API 키를 복사하세요.
wandb.login()

## ④ Baseline 학습

Sweep 비교용 기준 모델을 학습합니다.

In [ ]:
# Baseline 학습 실행
import sys
sys.path.insert(0, "/content/mlops_demo")

from train_baseline import main as run_baseline

print("🏁 Baseline 학습 시작...")
results = run_baseline()
print("✅ Baseline 학습 완료!")

## ⑤ WandB Sweep 실행

Bayesian 최적화로 하이퍼파라미터를 자동 탐색합니다.

| 파라미터 | 탐색 범위 |
|---|---|
| `batch` | 8, 16, 32 |
| `epochs` | 10, 15, 20 |
| `imgsz` | 320, 480, 640 |
| `lr0` | 0.0001 ~ 0.01 (uniform) |
| `weight_decay` | 0.0001 ~ 0.001 (uniform) |

In [ ]:
import wandb
import yaml

# sweep.yaml 로드
with open("/content/mlops_demo/sweep.yaml", "r") as f:
    sweep_config = yaml.safe_load(f)

# Sweep에서는 program 키가 CLI 전용이므로 제거
sweep_config.pop("program", None)

print("📋 Sweep 설정:")
print(yaml.dump(sweep_config, default_flow_style=False))

# Sweep 생성
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project="mlops_demo_yolo"
)
print(f"\n✅ Sweep 생성 완료!")
print(f"🔗 Sweep ID: {sweep_id}")
print(f"🌐 Sweep 링크: https://wandb.ai/<your-entity>/mlops_demo_yolo/sweeps/{sweep_id}")

In [ ]:
from train_sweep import main as sweep_agent_fn

# ── 실행할 Sweep 횟수 설정 ─────────────────────────────────────────────
# count: 총 몇 번의 run을 실행할지 결정합니다.
# Colab 무료 플랜의 경우 GPU 제한이 있으므로 3~5 권장
SWEEP_COUNT = 3  # ← 필요에 따라 수정

print(f"🤖 Sweep Agent 시작 (총 {SWEEP_COUNT}회 실행)...")
wandb.agent(
    sweep_id,
    function=sweep_agent_fn,
    count=SWEEP_COUNT
)
print("🎉 모든 Sweep 실행 완료!")

## ⑥ 결과 분석

In [ ]:
import wandb
import pandas as pd

# WandB API로 Sweep 결과 가져오기
api = wandb.Api()

# sweep_id가 위 셀에서 정의되어 있어야 합니다.
sweep = api.sweep(f"mlops_demo_yolo/{sweep_id}")
runs = sweep.runs

# 결과를 DataFrame으로 정리
records = []
for run in runs:
    record = {
        "run_name": run.name,
        "state": run.state,
        "mAP50-95": run.summary.get("metrics/mAP50-95(B)", None),
        "mAP50": run.summary.get("metrics/mAP50(B)", None),
    }
    record.update(run.config)
    records.append(record)

df = pd.DataFrame(records)
df_sorted = df.sort_values("mAP50-95", ascending=False).reset_index(drop=True)

print("🏆 Sweep 결과 (mAP50-95 기준 정렬):")
display(df_sorted)

In [ ]:
# 최적 하이퍼파라미터 출력
if not df_sorted.empty and df_sorted["mAP50-95"].notna().any():
    best = df_sorted.iloc[0]
    print("🥇 최적 하이퍼파라미터 조합:")
    print(f"   Run Name    : {best['run_name']}")
    print(f"   mAP50-95    : {best['mAP50-95']:.4f}")
    print(f"   mAP50       : {best.get('mAP50', 'N/A')}")
    print(f"   lr0         : {best.get('lr0', 'N/A')}")
    print(f"   weight_decay: {best.get('weight_decay', 'N/A')}")
    print(f"   batch       : {best.get('batch', 'N/A')}")
    print(f"   epochs      : {best.get('epochs', 'N/A')}")
    print(f"   imgsz       : {best.get('imgsz', 'N/A')}")
else:
    print("⚠️ 완료된 run이 없거나 mAP50-95 값이 없습니다.")